In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\prashant yadav\OneDrive\Desktop\IDS_project\data\campus_admission_raw_2000.csv")

In [4]:
df.shape

(2020, 26)

In [6]:
df_clean = df.copy()

print("Backup copy created.")

Backup copy created.


In [7]:
df.head()

,Student_ID,Application_ID,First_Name,Last_Name,Gender,Age,City,Category,10th_Percentage,12th_Percentage,...,Application_Status,Eligibility_Status,Document_Status,Verification_Date,Merit_Rank,Seat_Status,Fee_Status,Admission_Status,Admission_Date,Processing_Days
0,STU00530,APP00530,Arjun,Pawar,Female,18,Vasai,Open,73.23,97.00,...,Submitted,Eligible,Verified,2026-02-19,1362.0,Allocated,Paid,Confirmed,2026-02-24,16.0
1,STU00495,APP00495,Neha,Jadhav,Male,18,Virar,Open,90.12,63.39,...,Submitted,Not Eligible,Pending,NaN,NaN,Not Allocated,Not Paid,Not Confirmed,NaN,NaN
2,STU00395,APP00395,Tanmay,Joshi,Female,19,Andheri,SC,70.74,59.07,...,Submitted,Not Eligible,Incomplete,NaN,NaN,Not Allocated,Not Paid,Not Confirmed,NaN,NaN
3,STU01865,APP01865,Dev,Shinde,Male,21,Borivali,SC,76.66,50.34,...,Submitted,Not Eligible,Verified,2026-02-26,NaN,Not Allocated,Not Paid,Not Confirmed,NaN,NaN
4,STU01491,APP01491,Dev,Kulkarni,Female,18,Thane,Open,56.60,83.00,...,Submitted,Eligible,Incomplete,NaN,1337.0,Not Allocated,Not Paid,Not Confirmed,NaN,NaN


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Student_ID          2020 non-null   str    
 1   Application_ID      2020 non-null   str    
 2   First_Name          2020 non-null   str    
 3   Last_Name           2020 non-null   str    
 4   Gender              2020 non-null   str    
 5   Age                 2020 non-null   int64  
 6   City                1994 non-null   str    
 7   Category            2020 non-null   str    
 8   10th_Percentage     2020 non-null   float64
 9   12th_Percentage     2020 non-null   float64
 10  PCM_Percentage      1969 non-null   float64
 11  CET_Score           2020 non-null   float64
 12  CET_Percentile      1980 non-null   float64
 13  Course              2020 non-null   str    
 14  Application_Date    2020 non-null   str    
 15  Enquiry_Source      2020 non-null   str    
 16  Application_Statu

In [9]:
df.describe()

,Age,10th_Percentage,12th_Percentage,PCM_Percentage,CET_Score,CET_Percentile,Merit_Rank,Processing_Days
count,2020.000000,2020.000000,2020.000000,1969.000000,2020.000000,1980.000000,1100.000000,610.000000
mean,18.542574,77.752322,76.054510,76.093845,69.622693,73.486475,1789.228182,13.814754
std,0.926560,8.847934,10.209421,11.551560,10.756169,9.032939,534.738392,4.739186
min,17.000000,51.600000,45.000000,-5.000000,-10.000000,36.500000,108.000000,3.000000
25%,18.000000,71.525000,69.190000,68.500000,63.440000,67.250000,1436.500000,11.000000
50%,18.000000,77.685000,76.390000,76.440000,70.005000,73.815000,1807.000000,14.000000
75%,19.000000,84.180000,83.237500,84.010000,76.292500,79.580000,2156.500000,17.000000
max,21.000000,98.000000,97.000000,125.000000,120.000000,99.990000,3216.000000,25.000000


In [10]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
})

missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Count",
    ascending=False
)

,Missing_Count
Processing_Days,1410
Admission_Date,1410
Merit_Rank,920
Verification_Date,874
PCM_Percentage,51
CET_Percentile,40
City,26


In [11]:
duplicate_count = df[
    "Application_ID"
].duplicated().sum()

print(
    "Duplicate Application IDs:",
    duplicate_count
)

Duplicate Application IDs: 20


In [15]:
df_clean = df.drop_duplicates(
    subset="Application_ID",
    keep="first"
).reset_index(drop=True)

print(
    "Rows after removing duplicates:",
    len(df_clean)
)

Rows after removing duplicates: 2000


In [16]:
gender_mapping = {
    "male": "Male",
    "MALE": "Male",
    "M": "Male",
    "female": "Female",
    "FEMALE": "Female",
    "F": "Female"
}

df_clean["Gender"] = (
    df_clean["Gender"]
    .replace(gender_mapping)
)

In [17]:
df_clean["Gender"].value_counts(
    dropna=False
)

Gender
Male      1009
Female     991
Name: count, dtype: int64

In [19]:
df_clean["Category"].value_counts(
    dropna=False
)

Category
Open    1066
OBC      383
SC       237
EWS      189
ST       125
Name: count, dtype: int64[pyarrow]

In [20]:
invalid_pcm = (
    (df_clean["PCM_Percentage"] < 0) |
    (df_clean["PCM_Percentage"] > 100)
)

print(
    "Invalid PCM records:",
    invalid_pcm.sum()
)

Invalid PCM records: 15


In [21]:
df_clean.loc[
    invalid_pcm,
    "PCM_Percentage"
] = np.nan

In [23]:
pcm_median = df_clean[
    "PCM_Percentage"
].median()

print(
    "PCM Median:",
    round(pcm_median, 2)
)

PCM Median: 76.41


In [24]:
df_clean["PCM_Percentage"] = (
    df_clean["PCM_Percentage"]
    .fillna(pcm_median)
)

In [25]:
print(
    "Remaining missing PCM values:",
    df_clean["PCM_Percentage"].isnull().sum()
)

Remaining missing PCM values: 0


### **Clean CET Score**

In [26]:
invalid_cet = (
    (df_clean["CET_Score"] < 0) |
    (df_clean["CET_Score"] > 100)
)

print(
    "Invalid CET scores:",
    invalid_cet.sum()
)

Invalid CET scores: 10


In [27]:
df_clean.loc[
    invalid_cet,
    "CET_Score"
] = np.nan

In [28]:
cet_median = df_clean[
    "CET_Score"
].median()

df_clean["CET_Score"] = (
    df_clean["CET_Score"]
    .fillna(cet_median)
)

### Clean Dates

In [32]:
df_clean["Application_Date"] = pd.to_datetime(
    df_clean["Application_Date"],
    errors="coerce",
    dayfirst=True
)

In [34]:
df_clean["Verification_Date"] = pd.to_datetime(
    df_clean["Verification_Date"],
    errors="coerce",
    dayfirst=True
)

In [38]:
df_clean["Admission_Date"] = pd.to_datetime(
    df_clean["Admission_Date"],
    errors="coerce",
    dayfirst=True
)

In [39]:
df_clean[
    [
        "Application_Date",
        "Verification_Date",
        "Admission_Date"
    ]
].head()

,Application_Date,Verification_Date,Admission_Date
0,2026-08-02,2026-02-19,2026-02-24
1,NaT,NaT,NaT
2,2026-12-04,NaT,NaT
3,2026-11-02,2026-02-26,NaT
4,NaT,NaT,NaT


In [43]:
city_mode = df_clean["City"].mode()[0]

print("Most common city:", city_mode)

Most common city: Thane


In [44]:
df_clean["City"] = (
    df_clean["City"]
    .fillna(city_mode)
)

In [46]:
df_clean["Eligibility_Status"] = np.where(
    (
        (df_clean["PCM_Percentage"] >= 75) &
        (df_clean["CET_Percentile"] >= 60)
    ),
    "Eligible",
    "Not Eligible"
)
df_clean["Eligibility_Status"].value_counts()

Eligibility_Status
Eligible        1083
Not Eligible     917
Name: count, dtype: int64

In [51]:
print("Gender values:")
print(df_clean["Gender"].unique())

print("\nCategory values:")
print(df_clean["Category"].unique())

print("\nCourse values:")
print(df_clean["Course"].unique())

Gender values:
<ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str

Category values:
<ArrowStringArray>
['Open', 'SC', 'EWS', 'OBC', 'ST']
Length: 5, dtype: string

Course values:
<ArrowStringArray>
['Artificial Intelligence & Data Science',
                   'Computer Engineering',
                'Electronics Engineering',
                 'Mechanical Engineering',
                 'Information Technology']
Length: 5, dtype: str


### Save Clean Data

In [52]:
output_file = "campus_admission_cleaned_2000.csv"

df_clean.to_csv(
    output_file,
    index=False
)

print(
    f"Clean dataset saved successfully as: "
    f"{output_file}"
)

Clean dataset saved successfully as: campus_admission_cleaned_2000.csv
